## 1. Индекс файлов

Собираем таблицу `path, dataset, split, label` (0 – real, 1 – fake): 140k целиком, OpenForensics – подвыборка 10 тыс.

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from src.config import RAW_DIR
from src.data_index import build_index

index, mismatches = build_index()
index.head()

,path,dataset,split,label
0,140k/real_vs_fake/real-vs-fake/train/real/3135...,140k,train,0
1,140k/real_vs_fake/real-vs-fake/train/real/0288...,140k,train,0
2,140k/real_vs_fake/real-vs-fake/train/real/3398...,140k,train,0
3,140k/real_vs_fake/real-vs-fake/train/real/5387...,140k,train,0
4,140k/real_vs_fake/real-vs-fake/train/real/2414...,140k,train,0


Метка 140k берётся из папки `real/fake`. Расхождения с `label_str` и с `label` из CSV (там 1 – real, 0 – fake):

In [2]:
mismatches

{'label_str': 0, 'label': 0}

Проверки: пути существуют, нет повторов, число файлов на диске.

In [3]:
missing = (~index["path"].map(lambda p: (RAW_DIR / p).exists())).sum()
duplicates = index["path"].duplicated().sum()
files_140k = sum(1 for _ in (RAW_DIR / "140k").rglob("*.jpg"))
files_of = sum(1 for _ in (RAW_DIR / "openforensics").rglob("*.jpg"))
rows = index["dataset"].value_counts()

print("нет на диске:", missing)
print("повторы путей:", duplicates)
print("140k: строк", rows["140k"], "файлов", files_140k)
print("openforensics: файлов", files_of, "в подвыборке", rows["openforensics"])

нет на диске: 0
повторы путей: 0
140k: строк 140000 файлов 140000
openforensics: файлов 190335 в подвыборке 10000


In [4]:
pd.crosstab([index["dataset"], index["split"]], index["label"], margins=True)

label                    0      1     All
dataset       split                      
140k          test   10000  10000   20000
              train  50000  50000  100000
              valid  10000  10000   20000
openforensics test     286    286     572
              train   3678   3678    7356
              valid   1036   1036    2072
All                  75000  75000  150000

**Вывод:** метки 140k из папок совпадают с CSV, все пути на месте, повторов нет.

| показатель | 140k | OpenForensics |
|---|---|---|
| файлов на диске | 140 000 | 190 335 |
| в индексе | 140 000 | 10 000 |
| train / valid / test | 100 000 / 20 000 / 20 000 | 7 356 / 2 072 / 572 |
| real / fake | 70 000 / 70 000 | 5 000 / 5 000 |
| расхождений метки с CSV | 0 | – |
| отсутствующих путей / повторов | 0 / 0 | 0 / 0 |